# F1 Telemetry EDA — 2024 Bahrain GP (Race)

Exploring the telemetry pulled by `f1telemetry.data.ingest` and trying out the
baseline anomaly model from `f1telemetry.models.baseline` on real data.

Run `python -m f1telemetry.data.ingest 2024 Bahrain --session R` first if
`data/raw/telemetry.parquet` doesn't exist yet.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from f1telemetry.features.build_features import build_telemetry_features
from f1telemetry.models.baseline import BaselineAnomalyModel
from f1telemetry.data.ingest import load_session

%matplotlib inline

## 1. Load and sanity-check the telemetry

Before trusting anything downstream, check that the pulled data looks like real F1 telemetry: plausible top speeds, expected drivers, no obviously broken columns.

In [ ]:
telemetry = pd.read_parquet("../data/raw/telemetry.parquet")
print(telemetry.shape)
telemetry.head()

In [ ]:
top_speed_by_driver = (
    telemetry.groupby("Driver")["Speed"].max().sort_values(ascending=False)
)
top_speed_by_driver

Top speeds in the ~310-340 km/h range are expected for Bahrain; anything far outside that would suggest a data or units problem worth investigating before going further.

## 2. Baseline anomaly detection on real data

This is the first time we run the feature engineering + Isolation Forest baseline on *real* telemetry rather than synthetic test data.

In [ ]:
features = build_telemetry_features(telemetry)
model = BaselineAnomalyModel().fit(features)
scored = model.score(features)

top_anomalies = scored.sort_values("anomaly_score", ascending=False).head(20)
top_anomalies

In [ ]:
driver_to_plot = top_anomalies["Driver"].mode()[0]
lap_telemetry = telemetry[telemetry["Driver"] == driver_to_plot].sort_values("Distance")
lap_scored = scored[scored["Driver"] == driver_to_plot].sort_values("Distance")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(lap_telemetry["Distance"], lap_telemetry["Speed"], label="Speed")
anomaly_points = lap_scored[lap_scored["is_anomaly"]]
ax.scatter(anomaly_points["Distance"],
           lap_telemetry.set_index("Distance").reindex(anomaly_points["Distance"], method="nearest")["Speed"],
           color="red", zorder=5, label="Flagged anomaly")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Speed (km/h)")
ax.set_title(f"{driver_to_plot} — Speed trace with flagged anomalies")
ax.legend()
plt.show()

Look at *where* the red points land: near heavy braking zones or DRS activation is expected
and not very interesting on its own. A flag with no obvious cause (mid-straight, no braking,
no gear change) is either a genuinely odd telemetry moment or a sign the features need work.

## 3. Cross-check against real race events

FastF1 also exposes `race_control_messages` (flags, safety cars, incidents) for the session.

**Important limitation:** `f1telemetry.data.ingest` currently only pulls each driver's
*fastest lap*, not the full race. Race control incidents happen on whatever lap they happen
on, so they won't line up with a fastest-lap anomaly unless the incident happened to occur
during that specific lap. A real "does the model catch real incidents" validation needs
full-session telemetry (all laps), which is a good candidate for tomorrow's LSTM work rather
than this baseline. For now, this section is just to see what kind of events exist in this
session and get familiar with the data — not to claim a validated correlation yet.

In [ ]:
session = load_session(2024, "Bahrain", "R")  # cached, should be fast
race_control = session.race_control_messages
race_control[["Time", "Category", "Message", "Flag", "Scope", "RacingNumber"]].head(30)

In [ ]:
# Messages that look like actual flags/incidents, as a starting point for tomorrow's
# full-session anomaly work.
incident_like = race_control[race_control["Category"].isin(["Flag", "SafetyCar", "Drs"])]
incident_like

## 4. Compare two drivers' telemetry

A quick visual gut-check: overlay Speed vs Distance for the two fastest drivers on their fastest laps. Differences in braking points and corner-exit speed should be visible.

In [ ]:
top_two = top_speed_by_driver.head(2).index.tolist()

fig, ax = plt.subplots(figsize=(11, 4))
for driver in top_two:
    lap = telemetry[telemetry["Driver"] == driver].sort_values("Distance")
    ax.plot(lap["Distance"], lap["Speed"], label=driver)

ax.set_xlabel("Distance (m)")
ax.set_ylabel("Speed (km/h)")
ax.set_title(f"Speed trace comparison: {top_two[0]} vs {top_two[1]}")
ax.legend()
plt.show()